In [ ]:
import requests
import pandas as pd
import time
import os

INPUT_CSV = "mit_oai_with_orcid.csv"
SAVE_DIR = "downloaded_pdfs"

os.makedirs(SAVE_DIR, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "MIT-OAI-Research-Bot/1.0 (your_email@example.com)"
})

def download_pdfs():
    df = pd.read_csv(INPUT_CSV)

    total = 0

    for idx, row in df.iterrows():

        # Получаем handle напрямую
        oai_id = row["oai_identifier"]
        handle = oai_id.split(":")[-1]

        success = False

        for seq in range(1, 4):

            pdf_url = f"https://dspace.mit.edu/bitstream/handle/{handle}/filename.pdf?sequence={seq}"

            filename = os.path.join(
                SAVE_DIR,
                f"{handle.replace('/', '_')}_seq{seq}.pdf"
            )

            if os.path.exists(filename):
                success = True
                break

            try:
                r = session.get(pdf_url, timeout=60)

                if r.status_code == 429:
                    print("429 Too Many Requests — ждём 5 минут")
                    time.sleep(300)
                    continue

                if r.status_code == 404:
                    continue

                # Проверяем, что это реально PDF
                content_type = r.headers.get("Content-Type", "")
                if r.status_code == 200 and "pdf" in content_type.lower() and len(r.content) > 1000:

                    with open(filename, "wb") as f:
                        f.write(r.content)

                    print(f"Скачано: {filename}")
                    total += 1
                    success = True
                    break

            except Exception as e:
                print("Ошибка:", e)
                time.sleep(5)

        if not success:
            print("PDF не найден для:", handle)

        time.sleep(1.5)  # politeness delay

    print("Всего скачано:", total)


if __name__ == "__main__":
    download_pdfs()

Скачано: downloaded_pdfs\1721.1_88042_seq1.pdf
Скачано: downloaded_pdfs\1721.1_116916_seq1.pdf
Скачано: downloaded_pdfs\1721.1_60898_seq1.pdf
Скачано: downloaded_pdfs\1721.1_93078_seq2.pdf
429 Too Many Requests — ждём 5 минут
PDF не найден для: 1721.1/110238
Скачано: downloaded_pdfs\1721.1_76668_seq1.pdf
429 Too Many Requests — ждём 5 минут
PDF не найден для: 1721.1/116417
Скачано: downloaded_pdfs\1721.1_103534_seq1.pdf
Ошибка: HTTPSConnectionPool(host='dspace.mit.edu', port=443): Read timed out. (read timeout=60)
Ошибка: HTTPSConnectionPool(host='dspace.mit.edu', port=443): Read timed out. (read timeout=60)
Ошибка: HTTPSConnectionPool(host='dspace.mit.edu', port=443): Read timed out. (read timeout=60)
PDF не найден для: 1721.1/85999
Ошибка: HTTPSConnectionPool(host='dspace.mit.edu', port=443): Read timed out. (read timeout=60)
PDF не найден для: 1721.1/110505
Ошибка: HTTPSConnectionPool(host='dspace.mit.edu', port=443): Read timed out. (read timeout=60)
Ошибка: HTTPSConnectionPool(hos

KeyboardInterrupt: 